# Aula 8 · Parte 2 — Notebook 3: Métricas, desbalanceamento e vazamento

**Aprendizado de Máquina (PEPAPRM)** · IFSP Câmpus Presidente Epitácio

Objetivo: reconhecer dois erros clássicos de avaliação — a **acurácia que engana** em classes desbalanceadas e o **vazamento de dados** — e ver como evitá-los.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, KFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEMENTE = 42
pd.set_option("display.width", 100)

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)

## 1. A acurácia engana: classes desbalanceadas

Problema tipo *detecção de fraude*: só **5%** dos exemplos são a classe rara (1).

In [2]:
X_r, y_r = make_classification(n_samples=1000, n_features=6, n_informative=4, n_redundant=0,
                               n_clusters_per_class=1, weights=[0.95, 0.05], flip_y=0.0,
                               class_sep=1.0, random_state=SEMENTE)
print(f"{len(y_r)} exemplos, {y_r.mean() * 100:.1f}% são a classe rara")
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.30, random_state=SEMENTE, stratify=y_r)

candidatos = {
    "Baseline (sempre 'normal')": DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": LogisticRegression(max_iter=1000),
    "Reg. Logística (class_weight='balanced')": LogisticRegression(max_iter=1000, class_weight="balanced"),
}
linhas = []
for nome, modelo in candidatos.items():
    y_prev = modelo.fit(X_tr, y_tr).predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, y_prev).ravel()
    linhas.append((nome, accuracy_score(y_te, y_prev), precision_score(y_te, y_prev, zero_division=0),
                   recall_score(y_te, y_prev), f1_score(y_te, y_prev), fn, fp))
pd.DataFrame(linhas, columns=["modelo", "acurácia", "precisão", "recall", "f1",
                              "fraudes_perdidas (FN)", "alarmes_falsos (FP)"]).set_index("modelo").round(3)

1000 exemplos, 5.0% são a classe rara


,acurácia,precisão,recall,f1,fraudes_perdidas (FN),alarmes_falsos (FP)
modelo,,,,,,
Baseline (sempre 'normal'),0.950,0.000,0.000,0.000,15,0
Regressão Logística,0.977,0.900,0.600,0.720,6,1
Reg. Logística (class_weight='balanced'),0.940,0.452,0.933,0.609,1,17


**Como ler.** O baseline "nunca há fraude" tem 95% de acurácia sem aprender **nada** (recall zero). A regressão logística comum sobe para ~97,7%, mas ainda deixa passar 40% das fraudes (recall 0,60). Com `class_weight="balanced"`, a acurácia **cai** (~94%, abaixo do baseline!), porém o recall vai a ~93%: o modelo pega quase todas as fraudes ao custo de mais alarmes falsos. Qual é "melhor" depende do **custo de cada erro**.

| Métrica | Pergunta que responde | Priorize quando… |
|---|---|---|
| Precisão | dos que o modelo marcou como positivos, quantos eram? | alarme falso é caro |
| Recall | dos positivos reais, quantos o modelo achou? | perder um positivo é caro |
| F1 | equilíbrio entre precisão e recall | os dois erros importam |

## 2. Vazamento de dados

Cenário: 60 exemplos, 2000 atributos de **puro ruído** e alvo sorteado. Não existe padrão a aprender — a acurácia honesta é ~0,50.

- **Errado:** selecionar os 20 melhores atributos usando **todas** as linhas e só depois validar.
- **Certo:** colocar a seleção **dentro** do `Pipeline`, refeita em cada dobra.

In [3]:
errado, certo = [], []
for semente in range(20):
    rng = np.random.default_rng(semente)
    X_ruido = rng.normal(size=(60, 2000))
    y_ruido = rng.integers(0, 2, 60)
    dobras = StratifiedKFold(5, shuffle=True, random_state=SEMENTE)

    # usa TODAS as linhas
    X_sel = SelectKBest(f_classif, k=20).fit_transform(X_ruido, y_ruido)
    errado.append(cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_ruido, cv=dobras).mean())

    # Maneira correta
    pipe = Pipeline([("sel", SelectKBest(f_classif, k=20)), ("modelo", LogisticRegression(max_iter=1000))])
    certo.append(cross_val_score(pipe, X_ruido, y_ruido, cv=dobras).mean())

print(f"seleção ANTES de dividir (vazamento) : acurácia média = {np.mean(errado):.3f}")
print(f"seleção DENTRO do pipeline (correto) : acurácia média = {np.mean(certo):.3f}")

seleção ANTES de dividir (vazamento) : acurácia média = 0.915
seleção DENTRO do pipeline (correto) : acurácia média = 0.526


**Como ler.** O processo errado reporta ~91% de acurácia num problema onde não há nada a aprender! A seleção de atributos olhou o alvo de **todas** as linhas, inclusive das que depois serviram de validação: a informação do teste "vazou" para o treino. O mesmo vale para imputação, padronização e qualquer passo que aprende algo dos dados — deve ser ajustado **só com o treino** de cada dobra. Por isso o pré-processamento fica dentro do `Pipeline`.

### Para praticar
1. No bloco 1, calcule o custo total `10*FN + 1*FP` de cada modelo. Qual tem o menor custo?
2. No bloco 2, troque `k=20` por `k=5`. O vazamento diminui? Por quê?